In [34]:
import torch
import torch.nn as nn
import math

In [130]:
class SqueezeExcite(nn.Module):
    def __init__(self, in_channels, squeeze_channels):
        super(SqueezeExcite, self).__init__()
        self.se = nn.Sequential(
                        nn.AdaptiveAvgPool2d(1),
                        nn.Conv2d(in_channels, squeeze_channels, 1),
                        nn.SiLU(inplace=True),
                        nn.Conv2d(squeeze_channels, in_channels, 1),
                        nn.Sigmoid()
        )

    def forward(self,x):
        return x * self.se(x)                                

In [147]:
class MBConv(nn.Module):
    def __init__(self, expansion, in_channels, out_channels, stride, kernel, se_ratio=0.25):
        super(MBConv, self).__init__()
        hidden = in_channels * expansion
        self.use_residual = (stride == 1 and in_channels == out_channels)
        layers = []

        # Expansion
        if expansion != 1:
            layers += [nn.Conv2d(in_channels, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.SiLU(inplace=True)]
        else:
            hidden = in_channels

        # Depthwise
        layers += [nn.Conv2d(hidden, hidden, kernel, stride, padding=kernel//2, groups=hidden, bias=False), nn.BatchNorm2d(hidden), nn.SiLU(inplace=True)]
        # SE
        squeeze_channels  = int(max(1, in_channels * se_ratio))
        layers.append(SqueezeExcite(hidden, squeeze_channels))
        # Projection (linear bottleneck)
        layers += [nn.Conv2d(hidden, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels)]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        if(self.use_residual):
            out = out + x

        return out

In [148]:
@staticmethod
def adjust_width(channel, width_mult, divisor=8):
    channel *= width_mult
    return int(math.ceil(channel / divisor) * divisor)
@staticmethod
def adjust_depth(r, depth_mult):
    return int(math.ceil(r*depth_mult))

In [149]:
class EfficientNet(nn.Module):
    B0_CONFIG = [
        # expansion, out_channels, repeats, stride, kernel
        (1,  16, 1, 1, 3),
        (6,  24, 2, 2, 3),
        (6,  40, 2, 2, 5),
        (6,  80, 3, 2, 3),
        (6, 112, 3, 1, 5),
        (6, 192, 4, 2, 5),
        (6, 320, 1, 1, 3),
    ]

    def __init__(self, phi=0, num_classes=1000):
        super(EfficientNet, self).__init__()

        # Scaling coefficients
        beta = 1.1 # width
        alpha = 1.2 # depth

        depth_mult = alpha ** phi
        width_mult = beta ** phi

        # Stem
        in_channels = adjust_width(32, width_mult)
        self.stem = nn.Sequential(
                        nn.Conv2d(3, in_channels, 3, stride=2, padding=1, bias=False),
                        nn.BatchNorm2d(in_channels),
                        nn.SiLU(inplace=True)
        )

        blocks = []
        for expand, out_channels, depth, stride, k in self.B0_CONFIG:
            out_channels = adjust_width(out_channels, width_mult)
            depth = adjust_depth(depth, depth_mult)

            for i in range(depth):
                blocks.append(MBConv(expansion=expand, in_channels=in_channels, out_channels=out_channels,
                                     stride=stride if i == 0 else 1,
                                     kernel=k
                                    )
                             )
                in_channels = out_channels
        self.blocks = nn.Sequential(*blocks)

        head_channel = 1280#adjust_width(1280, width_mult)
        self.head = nn.Sequential(
                    nn.Conv2d(in_channels, head_channel, 1, bias=False),
                    nn.BatchNorm2d(head_channel),
                    nn.SiLU(inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(head_channel, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.pool(x).flatten(1)
        out = self.classifier(x)
        return out
        

In [150]:
model = EfficientNet(0)

In [151]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


Trainable parameters: 5,288,548


In [96]:
total = 0
for name, p in model.named_parameters():
    print(f"{name:50s} {p.numel():8d}")
    total += p.numel()

print("TOTAL:", total)


stem.0.weight                                           864
stem.1.weight                                            32
stem.1.bias                                              32
blocks.0.block.0.weight                                 288
blocks.0.block.1.se.1.weight                            256
blocks.0.block.1.se.3.weight                            256
blocks.0.block.2.weight                                 512
blocks.0.block.3.weight                                  16
blocks.0.block.3.bias                                    16
blocks.1.block.0.weight                                1536
blocks.1.block.1.weight                                  96
blocks.1.block.1.bias                                    96
blocks.1.block.3.weight                                 864
blocks.1.block.4.se.1.weight                           2304
blocks.1.block.4.se.3.weight                           2304
blocks.1.block.5.weight                                2304
blocks.1.block.6.weight                 

In [145]:
model

EfficientNet(
  (stem): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): MBConv(
      (block): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
        (3): SqueezeExcite(
          (se): Sequential(
            (0): AdaptiveAvgPool2d(output_size=1)
            (1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (2): SiLU(inplace=True)
            (3): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (4): Sigmoid()
          )
        )
        (4): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (5): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=Tru

In [137]:
from torchvision.models import efficientnet_b0
model = efficientnet_b0()
sum(p.numel() for p in model.parameters())


5288548

In [138]:
total = 0
for name, p in model.named_parameters():
    print(f"{name:50s} {p.numel():8d}")
    total += p.numel()

print("TOTAL:", total)

features.0.0.weight                                     864
features.0.1.weight                                      32
features.0.1.bias                                        32
features.1.0.block.0.0.weight                           288
features.1.0.block.0.1.weight                            32
features.1.0.block.0.1.bias                              32
features.1.0.block.1.fc1.weight                         256
features.1.0.block.1.fc1.bias                             8
features.1.0.block.1.fc2.weight                         256
features.1.0.block.1.fc2.bias                            32
features.1.0.block.2.0.weight                           512
features.1.0.block.2.1.weight                            16
features.1.0.block.2.1.bias                              16
features.2.0.block.0.0.weight                          1536
features.2.0.block.0.1.weight                            96
features.2.0.block.0.1.bias                              96
features.2.0.block.1.0.weight           

In [139]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [110]:
from torchvision.ops.misc import Conv2dNormActivation, SqueezeExcitation

In [111]:
help(SqueezeExcitation)

Help on class SqueezeExcitation in module torchvision.ops.misc:

class SqueezeExcitation(torch.nn.modules.module.Module)
 |  SqueezeExcitation(
 |      input_channels: int,
 |      squeeze_channels: int,
 |      activation: Callable[..., torch.nn.modules.module.Module] = <class 'torch.nn.modules.activation.ReLU'>,
 |      scale_activation: Callable[..., torch.nn.modules.module.Module] = <class 'torch.nn.modules.activation.Sigmoid'>
 |  ) -> None
 |
 |  This block implements the Squeeze-and-Excitation block from https://arxiv.org/abs/1709.01507 (see Fig. 1).
 |  Parameters ``activation``, and ``scale_activation`` correspond to ``delta`` and ``sigma`` in eq. 3.
 |
 |  Args:
 |      input_channels (int): Number of channels in the input image
 |      squeeze_channels (int): Number of squeeze channels
 |      activation (Callable[..., torch.nn.Module], optional): ``delta`` activation. Default: ``torch.nn.ReLU``
 |      scale_activation (Callable[..., torch.nn.Module]): ``sigma`` activation.